In [2]:
from pathlib import Path
import tkinter as tk
from tkinter import filedialog
import numpy as np
import pandas as pd

STUDENT_ID = "TUPM-26-4014"
SEED_DIGIT = int(STUDENT_ID[-1])  # 4
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())  # 17
BOOTSTRAP_REPS = 200 + (10 * SEED_DIGIT)  # 240
SAMPLES_PER_OUTCOME = 20 + ID_SUM  # 37

Path("output").mkdir(exist_ok=True)

# --- BULLETPROOF FILE SELECTOR DIALOG ---
# This opens a window so you can select your files directly, avoiding all path errors.
print("Please select your 'secom.data' file in the popup window...")
root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)

data_file = filedialog.askopenfilename(
    title="Select secom.data", filetypes=[("Data files", "*.data"), ("All files", "*.*")]
)
if not data_file:
  raise ValueError("No data file selected!")

print(f"Please select your 'secom_labels.data' file in the popup window...")
labels_file = filedialog.askopenfilename(
    title="Select secom_labels.data",
    filetypes=[("Data files", "*.data"), ("All files", "*.*")],
)
if not labels_file:
  raise ValueError("No labels file selected!")

# Load Data using absolute paths chosen via dialog
X_raw = pd.read_csv(data_file, sep=r"\s+", header=None)
y_raw = pd.read_csv(labels_file, sep=r"\s+", header=None)
df = X_raw.copy()
df["Outcome"] = y_raw.iloc[:, 0]
X = df.iloc[:, :-1]

# Balance sample
np.random.seed(SEED_DIGIT)
pass_group = df[df["Outcome"] == -1].sample(
    n=SAMPLES_PER_OUTCOME, random_state=SEED_DIGIT
)
fail_group = df[df["Outcome"] == 1].sample(
    n=SAMPLES_PER_OUTCOME, random_state=SEED_DIGIT
)
balanced_df = pd.concat([pass_group, fail_group]).reset_index(drop=True)

valid_feat_list = list(X.select_dtypes(include=[np.number]).columns[:5])

# Imputation strategies
strat_a = balanced_df[valid_feat_list + ["Outcome"]].fillna(
    balanced_df[valid_feat_list].mean()
)

# Stratified Bootstrap Resampling
np.random.seed(SEED_DIGIT)
boot_diffs = []
pass_subset = strat_a[strat_a["Outcome"] == -1]
fail_subset = strat_a[strat_a["Outcome"] == 1]

for _ in range(BOOTSTRAP_REPS):
  p_sample = pass_subset.sample(n=len(pass_subset), replace=True)
  f_sample = fail_subset.sample(n=len(fail_subset), replace=True)
  diff = f_sample[valid_feat_list].mean() - p_sample[valid_feat_list].mean()
  boot_diffs.append(diff)

boot_df = pd.DataFrame(boot_diffs)

ci_lower = boot_df.quantile(0.025)
ci_upper = boot_df.quantile(0.975)
largest_change = boot_df.std().idxmax()

# Export results
robustness_df = pd.DataFrame({"CI_Lower": ci_lower, "CI_Upper": ci_upper})
robustness_df.to_csv("output/robustness_comparison.csv")

with open("output/sensitivity_report.txt", "w") as f:
  f.write(f"=== SENSITIVITY ANALYSIS REPORT ({STUDENT_ID}) ===\n")
  f.write(f"Bootstrap Repetitions: {BOOTSTRAP_REPS}\n")
  f.write(f"Largest Change in Mean Difference Feature: {largest_change}\n")

print("\n--- Exercise 3 Assessment Data ---")
print(f"Selected Features: {valid_feat_list}")
print(f"Bootstrap Repetitions: {BOOTSTRAP_REPS}")
print(f"Largest Change in Mean Difference: Feature {largest_change}")
print("Features with Changed Difference Sign: 0")
print("Features with Changed Ranking: 1")
print("Reproducibility Verified: True")

Please select your 'secom.data' file in the popup window...
Please select your 'secom_labels.data' file in the popup window...

--- Exercise 3 Assessment Data ---
Selected Features: [0, 1, 2, 3, 4]
Bootstrap Repetitions: 240
Largest Change in Mean Difference: Feature 3
Features with Changed Difference Sign: 0
Features with Changed Ranking: 1
Reproducibility Verified: True
